<a href="https://colab.research.google.com/github/Sanjivkumar100/Emotion_Aware_Production_System/blob/main/emotion_and_prediction_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow
!pip install keras

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('stopwords')
stopwords=set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
sad_df=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/sadness-ratings-0to1.train.txt',sep='\t',header=None)
anger_df=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/anger-ratings-0to1.train.txt',sep='\t',header=None)
fear_df=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/fear-ratings-0to1.train.txt',sep='\t',header=None)
joy_df=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/joy-ratings-0to1.train.txt',sep='\t',header=None)

In [ ]:
cols=['id','text','emotion','intensity']
sad_df.columns=cols
anger_df.columns=cols
fear_df.columns=cols
joy_df.columns=cols

nrc_df=pd.concat([sad_df,anger_df,fear_df,joy_df])
nrc_df.reset_index(drop=True,inplace=True)
nrc_df.head()

,id,text,emotion,intensity
0,40000,Depression sucks! #depression,sadness,0.958
1,40001,Feeling worthless as always #depression,sadness,0.958
2,40002,Feeling worthless as always,sadness,0.958
3,40003,My #Fibromyalgia has been really bad lately wh...,sadness,0.946
4,40004,Im think ima lay in bed all day and sulk. Life...,sadness,0.934


In [ ]:
train_go=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/train.tsv',sep='\t')
test_go=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/test.tsv',sep='\t')
dev_go=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/dev.tsv',sep='\t')

go_df=pd.concat([train_go,test_go,dev_go])
go_df.reset_index(drop=True,inplace=True)
go_df.head()


,My favourite food is anything I didn't have to cook myself.,27,eebbqej,"I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!",25,eecwqtt,Is this in New Orleans?? I really feel like this is New Orleans.,edgurhb
0,"Now if he does off himself, everyone will thin...",27,ed00q6i,NaN,NaN,NaN,NaN,NaN
1,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj,NaN,NaN,NaN,NaN,NaN
2,To make her feel threatened,14,ed7ypvh,NaN,NaN,NaN,NaN,NaN
3,Dirty Southern Wankers,3,ed0bdzj,NaN,NaN,NaN,NaN,NaN
4,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,26,edvnz26,NaN,NaN,NaN,NaN,NaN


In [ ]:
train_go=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/train.tsv',sep='\t', header=None, names=['text', 'labels', 'id'])
test_go=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/test.tsv',sep='\t', header=None, names=['text', 'labels', 'id'])
dev_go=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/dev.tsv',sep='\t', header=None, names=['text', 'labels', 'id'])

go_df=pd.concat([train_go,test_go,dev_go])
go_df.reset_index(drop=True,inplace=True)



In [ ]:
go_df.head()

,text,labels,id
0,My favourite food is anything I didn't have to...,27,eebbqej
1,"Now if he does off himself, everyone will thin...",27,ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj
3,To make her feel threatened,14,ed7ypvh
4,Dirty Southern Wankers,3,ed0bdzj


In [ ]:
go_df = go_df[['text','labels']]
go_df["labels"] = go_df["labels"].astype(str)
go_df["primary_label"] = go_df["labels"].apply(lambda x: x.split(",")[0])
go_df["primary_label"] = go_df["primary_label"].astype(int)



In [ ]:
emotion_map = {
0:"admiration", 1:"amusement", 2:"anger", 3:"annoyance",
4:"approval", 5:"caring", 6:"confusion", 7:"curiosity",
8:"desire", 9:"disappointment", 10:"disapproval", 11:"disgust",
12:"embarrassment", 13:"excitement", 14:"fear", 15:"gratitude",
16:"grief", 17:"joy", 18:"love", 19:"nervousness",
20:"optimism", 21:"pride", 22:"realization", 23:"relief",
24:"remorse", 25:"sadness", 26:"surprise", 27:"neutral"
}
go_df["emotion"] = go_df["primary_label"].map(emotion_map)
go_df[["text","emotion"]].head()


,text,emotion
0,My favourite food is anything I didn't have to...,neutral
1,"Now if he does off himself, everyone will thin...",neutral
2,WHY THE FUCK IS BAYLESS ISOING,anger
3,To make her feel threatened,fear
4,Dirty Southern Wankers,annoyance


In [ ]:
go_df["intensity"] = 0.5


In [ ]:
emotion_df = pd.concat([nrc_df[['text','emotion','intensity']], go_df])
emotion_df.reset_index(drop=True, inplace=True)


In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    return text.strip()

emotion_df["clean_text"] = emotion_df["text"].apply(clean_text)

emotion_df.head()

,text,emotion,intensity,labels,primary_label,clean_text
0,Depression sucks! #depression,sadness,0.958,NaN,NaN,depression sucks
1,Feeling worthless as always #depression,sadness,0.958,NaN,NaN,feeling worthless as always
2,Feeling worthless as always,sadness,0.958,NaN,NaN,feeling worthless as always
3,My #Fibromyalgia has been really bad lately wh...,sadness,0.946,NaN,NaN,my has been really bad lately which is not go...
4,Im think ima lay in bed all day and sulk. Life...,sadness,0.934,NaN,NaN,im think ima lay in bed all day and sulk life ...


In [ ]:
def remove_stopwords(text):
  return " ".join([w for w in str(text).split() if w not in stopwords])

emotion_df['clean_text']= emotion_df['clean_text'].apply(remove_stopwords)

In [ ]:
MAX_WORDS = 15000
MAX_LEN = 120

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(emotion_df["clean_text"])

X = tokenizer.texts_to_sequences(emotion_df["clean_text"])
X = pad_sequences(X, maxlen=MAX_LEN)


In [ ]:
label_encoder= LabelEncoder()
y_class= label_encoder.fit_transform(emotion_df["emotion"])
scaler= MinMaxScaler()
y_reg= scaler.fit_transform(emotion_df[["intensity"]])

In [ ]:
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg,
    test_size=0.2,
    random_state=42
)


In [ ]:
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)

np.save("y_class_train.npy", y_class_train)
np.save("y_class_test.npy", y_class_test)

np.save("y_reg_train.npy", y_reg_train)
np.save("y_reg_test.npy", y_reg_test)


In [ ]:
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

with open("label_encoder.pkl","wb") as f:
    pickle.dump(label_encoder,f)


In [ ]:
product_df= pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/flipkart_com-ecommerce_sample.csv')
product_df.shape

(20000, 15)

In [ ]:
product_df=product_df[[
    'product_name',
    'product_category_tree',
    'description',
    'brand',
    'retail_price',
]]
product_df.dropna(inplace=True)

In [ ]:
def clean_category(cat):
  cat = cat.replace('[','').replace(']','').replace('"','')
  return cat.split('>>')[0].strip().lower()

product_df['category']= product_df['product_category_tree'].apply(clean_category)
product_df

,product_name,product_category_tree,description,brand,retail_price,category
0,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",Key Features of Alisha Solid Women's Cycling S...,Alisha,999.0,clothing
1,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",FabHomeDecor Fabric Double Sofa Bed (Finish Co...,FabHomeDecor,32157.0,furniture
2,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",Key Features of AW Bellies Sandals Wedges Heel...,AW,999.0,footwear
3,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",Key Features of Alisha Solid Women's Cycling S...,Alisha,699.0,clothing
4,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",Specifications of Sicons All Purpose Arnica Do...,Sicons,220.0,pet supplies
...,...,...,...,...,...,...
19995,WallDesign Small Vinyl Sticker,"[""Baby Care >> Baby & Kids Gifts >> Stickers >...",Buy WallDesign Small Vinyl Sticker for Rs.730 ...,WallDesign,1500.0,baby care
19996,Wallmantra Large Vinyl Stickers Sticker,"[""Baby Care >> Baby & Kids Gifts >> Stickers >...",Buy Wallmantra Large Vinyl Stickers Sticker fo...,Wallmantra,1429.0,baby care
19997,Elite Collection Medium Acrylic Sticker,"[""Baby Care >> Baby & Kids Gifts >> Stickers >...",Buy Elite Collection Medium Acrylic Sticker fo...,Elite Collection,1299.0,baby care
19998,Elite Collection Medium Acrylic Sticker,"[""Baby Care >> Baby & Kids Gifts >> Stickers >...",Buy Elite Collection Medium Acrylic Sticker fo...,Elite Collection,1499.0,baby care


In [ ]:
product_df['category'].unique()

array(['clothing', 'furniture', 'footwear', 'pet supplies',
       'eternal gandhi super series crystal paper weight...',
       'pens & stationery',
       'bengal blooms rose artificial plant  with pot (3...',
       'bags, wallets & belts', 'home decor & festive needs',
       'automotive', 'tools & hardware',
       "vishudh printed women's straight kurta",
       "vishudh printed women's anarkali kurta",
       'buildtrack pir wireless motion sensor - one swit...',
       'skayvon summersible single phase pump controller...',
       "masara solid women's straight kurta",
       'skayvon submersibble three phase pump controller...',
       'behringer xenyx 502 analog sound mixer',
       "noor embroidered women's straight kurta",
       "libas printed women's a-line kurta",
       "libas printed women's anarkali kurta", 'sports & fitness',
       'home furnishing', 'baby care', 'mobiles & accessories',
       'food & nutrition', 'toys & school supplies', 'jewellery',
       'cellba

In [ ]:
def clean_text(text):
  text= str(text).lower()
  text= re.sub(r"http\S+", "", text)
  text= re.sub(r"[^a-zA-Z\s]","",text)
  return text.strip()

def stopword_remove(text):
  return " ".join([w for w in str(text).split() if w not in stopwords])

product_df["clean_description"] = product_df["description"].apply(clean_text)
product_df["clean_description"] = product_df["clean_description"].apply(remove_stopwords)


In [ ]:
category_map = {

    "fashion": ["clothing", "footwear", "bags", "jewellery", "watches"],

    "electronics": ["mobiles", "computers", "gaming", "cameras", "home entertainment"],

    "home": ["home", "furniture", "kitchen", "home furnishing", "home decor"],

    "fitness": ["sports", "fitness"],

    "beauty": ["beauty", "personal care"],

    "kids": ["baby care", "toys"],

    "automotive": ["automotive"],

    "stationery": ["pens", "stationery"]
}

def map_category(cat):

    cat = str(cat).lower()

    for main_cat, keywords in category_map.items():
        if any(keyword in cat for keyword in keywords):
            return main_cat

    return "other"
product_df["normalized_category"] = product_df["category"].apply(map_category)



In [ ]:
product_df["normalized_category"].value_counts()


,count
normalized_category,
fashion,7230
home,2209
electronics,1795
automotive,1010
other,784
kids,558
beauty,199
stationery,174
fitness,111


In [ ]:
emotion_product_map = {

    "joy": ["fashion","electronics","fitness"],

    "sadness": ["home","beauty","stationery"],

    "anger": ["fitness","electronics"],

    "fear": ["home","beauty"]
}

def recommend_prodcuts(emotion):
  cats=emotion_product_map.get(emotion.lower(),[])
  return product_df[
      product_df["normalized_category"].isin(cats)
      ][["product_name","normalized_category","brand","retail_price"]]

In [ ]:
recommend_prodcuts("joy").head()


,product_name,normalized_category,brand,retail_price
0,Alisha Solid Women's Cycling Shorts,fashion,Alisha,999.0
2,AW Bellies,fashion,AW,999.0
3,Alisha Solid Women's Cycling Shorts,fashion,Alisha,699.0
6,Alisha Solid Women's Cycling Shorts,fashion,Alisha,1199.0
8,"dilli bazaaar Bellies, Corporate Casuals, Casuals",fashion,dilli bazaaar,699.0


In [ ]:
product_df.to_csv("processed_flipkart_products.csv", index=False)
